# Data Clean and Inference Result Extraction

For the [original csv](./all_providers_clean_data_llm_2026-01-16.csv), we apply the below preprocessing:
1. Filter out the GPU models of H100 and H200 families
2. Replace the "lambda_labs" in "provider" field with "lambda"
3. Confirm that there is no `gpu_id` with multiple provider
4. Extract the average token throughput of "128-128-con_1", "128-128-con_25", "128-128-con_50", "128-128-con_100" and 
"4096-512-con_1", "4096-512-con_25", "4096-512-con_50", "4096-512-con_100" from the field "inference results".

In [1]:
%pip install pandas matplotlib seaborn scipy --quiet


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
TARGET_GPU_MODELS = [
    "NVIDIA H100 80GB HBM3",
    "NVIDIA H200",
    "NVIDIA H100 PCIe"
]

PROVIDER_REPLACE_MAP = {
    "lambda_labs": "lambda",
    "lambda labs": "lambda"
}

In [3]:
import pandas as pd

df_raw = pd.read_csv('all_providers_clean_data_llm_2026-01-16.csv', sep=',')

In [4]:
# --- Replace provider names ---
df_raw['provider'] = df_raw['provider'].replace(PROVIDER_REPLACE_MAP)

In [5]:
# --- Identify gpu_id values mapped to >1 distinct provider
prov_counts = df_raw.groupby("gpu_id")["provider"].nunique(dropna=False)
bad_gpu_ids = prov_counts[prov_counts > 1].index

print(f"gpu_id with multiple providers: {len(bad_gpu_ids)}")

# Show the conflicting mappings (gpu_id -> providers + row counts per provider)
if len(bad_gpu_ids) > 0:
    conflicts = (
        df_raw[df_raw["gpu_id"].isin(bad_gpu_ids)]
        .groupby(["gpu_id", "provider"])
        .size()
        .reset_index(name="rows")
        .sort_values(["gpu_id", "rows"], ascending=[True, False])
    )
    display(conflicts)

gpu_id with multiple providers: 0


In [6]:
# --- Filter data for target GPU models ---
df_filtered = df_raw[df_raw['gpu_model'].isin(TARGET_GPU_MODELS)].copy() 

In [7]:
# --- Drop the gpu_id values with empty "inference results" entries ---
INF_RES_HEADER = "inference results"

df_filtered = df_filtered.dropna(subset=[INF_RES_HEADER])

display(df_filtered.tail(5))

print(f"Filtered data contains {df_filtered.shape[0]} records across {df_filtered['provider'].nunique()} providers and {df_filtered['gpu_model'].nunique()} GPU models.")  

,provider,gpu_model,gpu_id,fp32,fp16,bf16,mixed_precision_tflops,pcie_bandwidth,memory_bandwidth,job_id,create_date,accelerator_driver_version,cuda_version,fine-tuning results,inference results
131,lambda,NVIDIA H100 80GB HBM3,GPU-c8fc4e40-0099-7a60-364c-8de1e426da86,363.42,653.33,707.46,637.45,55.530,3018.93,2006490778305699840,2025-12-31 22:20:55.002694+00:00,570.195.03,12.8,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 82.454...
132,lambda,NVIDIA H100 80GB HBM3,GPU-8135026c-4a23-3114-3f5c-bbd3ea7e2599,337.13,686.09,718.64,618.93,55.495,3009.40,2006429384218648576,2025-12-31 18:16:57.511140+00:00,570.195.03,12.8,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 84.484...
133,lambda,NVIDIA H100 80GB HBM3,GPU-99b3691c-33fa-1ab0-598f-55ca8f628bdd,335.05,686.27,722.37,627.77,55.450,3021.24,2006143509383487488,2025-12-30 23:20:59.639062+00:00,570.195.03,12.8,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 75.431...
134,lambda,NVIDIA H100 80GB HBM3,GPU-99b3691c-33fa-1ab0-598f-55ca8f628bdd,334.84,684.02,735.82,620.75,55.450,3021.50,2006134414408556544,2025-12-30 22:44:51.227957+00:00,570.195.03,12.8,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 84.755...
172,lambda,NVIDIA H100 80GB HBM3,GPU-38038c13-6f1f-6c11-7141-6a0714657f36,362.61,667.50,703.16,638.20,55.490,3023.49,2009478857631080448,2026-01-09 04:14:28.629140+00:00,580.105.08,13.0,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 95.543...


Filtered data contains 88 records across 2 providers and 2 GPU models.


Extract the token throughput from field "inference results". A sample field is [here](./inference_results_sample.json).

In [8]:
import pandas as pd
import numpy as np
import ast


# ---- The extraction paths ----
def get_tps_metrics(data):
    return {
    "128_128_con_1_tps":   data["summary"]["chat"]["128_128"]["concurrency_1"]["output_token_throughput_avg"],
    "128_128_con_25_tps":  data["summary"]["chat"]["128_128"]["concurrency_25"]["output_token_throughput_avg"],
    "128_128_con_50_tps":  data["summary"]["chat"]["128_128"]["concurrency_50"]["output_token_throughput_avg"],
    "128_128_con_100_tps": data["summary"]["chat"]["128_128"]["concurrency_100"]["output_token_throughput_avg"],
    "4096_512_con_1_tps":  data["summary"]["summarization"]["4096_512"]["concurrency_1"]["output_token_throughput_avg"],
    "4096_512_con_25_tps": data["summary"]["summarization"]["4096_512"]["concurrency_25"]["output_token_throughput_avg"],
    "4096_512_con_50_tps": data["summary"]["summarization"]["4096_512"]["concurrency_50"]["output_token_throughput_avg"],
    "4096_512_con_100_tps":data["summary"]["summarization"]["4096_512"]["concurrency_100"]["output_token_throughput_avg"]
    }

In [9]:
# ---- test extraction on first valid sample ----
samples = df_filtered[INF_RES_HEADER].dropna().head(1).tolist()
for i, s in enumerate(samples):
    try:
        ast.literal_eval(s)
        print("ast.literal_eval OK")
        ret = get_tps_metrics(ast.literal_eval(s))
        print(ret)
    except Exception as e:
        print("ast.literal_eval FAIL ({type(e).__name__})")

ast.literal_eval OK
{'128_128_con_1_tps': 226.53389668231665, '128_128_con_25_tps': 4146.032247049465, '128_128_con_50_tps': 7202.0932415705065, '128_128_con_100_tps': 10841.97306179793, '4096_512_con_1_tps': 207.52924939059724, '4096_512_con_25_tps': 1953.6287584848453, '4096_512_con_50_tps': 2454.5046084254022, '4096_512_con_100_tps': 2797.364392502194}


In [10]:
import ast
import pandas as pd

def safe_get_tps(s):
    try:
        obj = ast.literal_eval(s)
        ret = get_tps_metrics(obj)
        # treat empty dict as invalid
        if not ret:
            return None
        return ret
    except Exception:
        return None

# Extract TPS metrics
metrics_series = (
    df_filtered[INF_RES_HEADER]
    .apply(lambda s: safe_get_tps(s) if pd.notna(s) else None)
)

# Expand into columns
metrics_df = metrics_series.apply(pd.Series)

display(metrics_df.head(5))


,128_128_con_1_tps,128_128_con_25_tps,128_128_con_50_tps,128_128_con_100_tps,4096_512_con_1_tps,4096_512_con_25_tps,4096_512_con_50_tps,4096_512_con_100_tps
0,226.533897,4146.032247,7202.093242,10841.973062,207.529249,1953.628758,2454.504608,2797.364393
1,222.914155,4044.706536,7098.091007,11009.809711,202.276765,1931.550259,2448.381142,2803.202924
2,221.616213,4061.401209,7035.072942,10810.394078,201.511195,1935.899597,2438.868274,2791.227785
3,223.591858,4084.558547,6997.933435,10748.996444,202.789170,1945.570375,2453.579486,2804.573343
4,222.217408,4062.314502,7181.631417,10759.002615,201.402104,1928.525672,2435.293396,2778.712181


In [11]:
# --- Combine with original dataframe ---
df_new = pd.concat([df_filtered.reset_index(drop=True), metrics_df.reset_index(drop=True)], axis=1)

In [12]:
display(df_new.head(5))

,provider,gpu_model,gpu_id,fp32,fp16,bf16,mixed_precision_tflops,pcie_bandwidth,memory_bandwidth,job_id,...,fine-tuning results,inference results,128_128_con_1_tps,128_128_con_25_tps,128_128_con_50_tps,128_128_con_100_tps,4096_512_con_1_tps,4096_512_con_25_tps,4096_512_con_50_tps,4096_512_con_100_tps
0,lambda,NVIDIA H100 80GB HBM3,GPU-38038c13-6f1f-6c11-7141-6a0714657f36,362.61,667.50,703.16,638.20,55.490,3023.49,2009478857631080448,...,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 95.543...,226.533897,4146.032247,7202.093242,10841.973062,207.529249,1953.628758,2454.504608,2797.364393
1,lambda,NVIDIA H100 80GB HBM3,GPU-fc201365-e354-e3c9-e303-e28ec5c965e6,361.67,664.39,704.65,643.82,55.410,3021.18,2008754148182466560,...,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 82.466...,222.914155,4044.706536,7098.091007,11009.809711,202.276765,1931.550259,2448.381142,2803.202924
2,lambda,NVIDIA H100 80GB HBM3,GPU-f6e0d80a-1ee4-6af1-6585-7b23e7185153,369.82,673.76,714.21,647.91,55.495,3013.94,2008391642800857088,...,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 84.497...,221.616213,4061.401209,7035.072942,10810.394078,201.511195,1935.899597,2438.868274,2791.227785
3,lambda,NVIDIA H100 80GB HBM3,GPU-226c1ecf-220c-e7c8-56b4-21bd0d45aa07,366.92,675.11,715.17,640.40,55.500,3023.26,2008031566604935168,...,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 90.511...,223.591858,4084.558547,6997.933435,10748.996444,202.789170,1945.570375,2453.579486,2804.573343
4,lambda,NVIDIA H100 80GB HBM3,GPU-8135026c-4a23-3114-3f5c-bbd3ea7e2599,363.21,649.95,696.75,635.46,55.500,3013.86,2007668331074691072,...,"{'tokens_per_step': 8192, 'tokens_per_second':...",{'timing': {'nim_startup_time_seconds': 99.535...,222.217408,4062.314502,7181.631417,10759.002615,201.402104,1928.525672,2435.293396,2778.712181


In [13]:
FIELDS_TO_KEEP = [
    'provider',
    'gpu_model',
    'gpu_id',
    'fp16',
    'memory_bandwidth',
    'create_date',
    'accelerator_driver_version',
    'cuda_version',
    '128_128_con_1_tps',
    '128_128_con_25_tps',
    '128_128_con_50_tps',
    '128_128_con_100_tps',
    '4096_512_con_1_tps',
    '4096_512_con_25_tps',
    '4096_512_con_50_tps',
    '4096_512_con_100_tps'
]

df_new = df_new[FIELDS_TO_KEEP]
df_new.to_csv('nv-llama3-4b_inf_processed_data.csv', index=False)